# Go2 速度策略训练：一键演示

本 notebook 从环境自检到训练出图走完一遍 `unitree_rl_mjx` 的训练链路：

1. 检查 JAX 后端（ROCm / CUDA / CPU）
2. 冒烟规模训练 Go2 平地速度任务（4000 万步，GPU 上约 20 分钟）
3. 画训练回报曲线
4. 渲染训练结果的行走动图

启动方式（在仓库根目录）：

```bash
uv run --with jupyter --with matplotlib --with imageio jupyter lab
```

安装见《教程 1：环境安装》。

In [ ]:
import os

# gfx1201 已知约束：训练必须禁用 XLA 命令缓冲。
# 只在 ROCm 上设置该 flag，其他后端保持默认行为。
import jax

backend = jax.default_backend()
devices = jax.devices()
print(f"backend: {backend}")
print(f"devices: {devices}")
IS_ROCM = any("rocm" in str(d).lower() for d in devices)
if backend == "cpu":
  print("警告：未检测到 GPU。CPU 也能跑，但 4000 万步会非常慢；"
        "建议把下方 NUM_TIMESTEPS 降到 2_000_000 只验证链路。")

In [ ]:
import subprocess
import sys

NUM_TIMESTEPS = 41_943_040
OUT_DIR = "runs/notebook-demo"

env = dict(os.environ)
if IS_ROCM:
  env["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="

proc = subprocess.Popen(
  [sys.executable, "-m", "unitree_rl_mjx.train.go2_velocity",
   "--seed", "0", "--num-timesteps", str(NUM_TIMESTEPS),
   "--out-dir", OUT_DIR],
  env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
  print(line, end="")
assert proc.wait() == 0, "training failed"

In [ ]:
import json

import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"{OUT_DIR}/metrics.jsonl")]
steps = [r["step"] for r in rows if "eval/episode_reward" in r]
reward = [r["eval/episode_reward"] for r in rows if "eval/episode_reward" in r]

plt.figure(figsize=(7, 4))
plt.plot(steps, reward, marker="o")
plt.xlabel("environment steps")
plt.ylabel("episode reward")
plt.title("Go2 velocity task, smoke-scale training")
plt.grid(alpha=0.3)
plt.show()
print(f"final reward: {reward[-1]:.1f}")

In [ ]:
# 无显示器的机器（云实例、SSH 服务器）用 EGL 做离屏渲染。
if "DISPLAY" not in os.environ:
  os.environ.setdefault("MUJOCO_GL", "egl")

import imageio
import mujoco
import numpy as np

from unitree_rl_mjx.envs import Go2VelocityFlat

qpos = np.load(f"{OUT_DIR}/trajectory.npz")["qpos"]
model = Go2VelocityFlat().mj_model
model.vis.global_.offwidth, model.vis.global_.offheight = 960, 540
data = mujoco.MjData(model)
camera = mujoco.MjvCamera()
camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
camera.trackbodyid = model.body("base_link").id
camera.distance, camera.elevation, camera.azimuth = 1.6, -18, 125

renderer = mujoco.Renderer(model, height=540, width=960)
frames = []
for q in qpos[::4]:
  data.qpos[:] = q
  mujoco.mj_forward(model, data)
  renderer.update_scene(data, camera)
  frames.append(renderer.render().copy())
renderer.close()

imageio.mimsave(f"{OUT_DIR}/rollout.gif", frames, fps=13)
print(f"wrote {OUT_DIR}/rollout.gif ({len(frames)} frames)")

from IPython.display import Image, display

display(Image(filename=f"{OUT_DIR}/rollout.gif"))

## 下一步

- 完整训练（9.8 亿步）：去掉 `--num-timesteps`，见《教程 2：训练》。
- 把策略装进官方 C++ 部署栈闭环：见《教程 3：sim2sim》。